In [107]:
# !pip install galois scipy ldpc

In [1]:
from scipy.special import binom, lambertw
from importlib.metadata import version
from qwen import *
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import time
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download
from constants import test_prompts
import numpy as np
from prc import KeyGen, Encode, Detect
from qwen import KVCache                                                                                                                                                                                                                           


pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]


for p in pkgs:
    print(f"{p} version: {version(p)}")

USE_BASE_MODEL = True
USE_REASONING_MODEL = False
USE_INSTRUCT_MODEL = False

if (USE_BASE_MODEL + USE_REASONING_MODEL
    + USE_INSTRUCT_MODEL) != 1:
    raise AttributeError("Only one of the options above can be True.")


CHOOSE_MODEL = "0.6B"
QWEN3_CONFIG = return_qwen_config(CHOOSE_MODEL)

model = Qwen3Model(QWEN3_CONFIG)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}"
else:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}-Base"

local_dir = Path(repo_id).parts[-1]

if CHOOSE_MODEL == "0.6B":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
else:
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device)
del weights_dict

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')


if USE_REASONING_MODEL:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}/tokenizer.json"
else:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}-Base/tokenizer.json"

hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

print(tokenizer_file_path)
if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=USE_REASONING_MODEL
    )

else:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=False,
        add_generation_prompt=False,
        add_thinking=False
    )


def prompt_to_ids(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    formatted_text = tok.apply_chat_template(messages, tokenize=False,     add_generation_prompt=True)
    return tokenizer.encode(formatted_text)


def detect(P, vec, z, entropies, entropy_threshold=0.5, fpr=1e-9):
    r, n = P.shape

    # 1. Mark reliable bit positions
    reliable = entropies >= entropy_threshold

    # 2. Drop any parity check that touches an unreliable bit
    P_int = np.asarray(P, dtype=np.int64)
    unreliable_hits = P_int[:, ~reliable].sum(axis=1)   # # of unreliable bits per check
    keep_check = unreliable_hits == 0
    r_eff = int(keep_check.sum())

    if r_eff == 0:
        return False

    # 3. Compute syndrome weight on the surviving checks
    syndrome = np.asarray(P @ (vec + z), dtype=np.int64) % 2
    wt = int(syndrome[keep_check].sum())

    # 4. Hoeffding threshold scaled to r_eff (NOT r)
    threshold = r_eff / 2 - np.sqrt(0.5 * r_eff * np.log(1 / fpr))

    return wt < threshold


vocab_size  = model.tok_emb.weight.shape[0]

v_0 = torch.zeros(vocab_size, dtype=torch.bfloat16).to(device)
indices  = torch.randperm(vocab_size)[:vocab_size//2]
v_0[indices] = 1.0
v_1 = 1-v_0
partition = torch.concat([v_0,v_1]).reshape(2, vocab_size)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


"""
PRC text watermarking: generate + detect (entropy-aware, calibrated).

Generation model (Christ-Gunn / Kuditipudi style)
-------------------------------------------------
- Encode(encoding_key) returns a +/-1 PRC codeword; eta noise is baked in
  via KeyGen's noise_rate.
- For each generated token:
    1. p = LM prob of partition 1.
    2. Codeword bit xi -> Bernoulli sampling parameter:
         bern_p = 2*xi*p              if p <= 0.5
         bern_p = 1 - 2*(1-xi)*(1-p)   if p > 0.5
    3. b ~ Bernoulli(bern_p) selects the partition.
    4. argmax token within that partition.
- The realized bit b matches xi only noisily. When p is far from 0.5, the
  observation is uninformative about xi -- this is the LPN noise channel.

Detection model (entropy-aware + null-calibrated)
-------------------------------------------------
- For each generated token we now also need the LM's partition probability
  p that was used to sample it. `generate_text_watermark_prc` returns these
  alongside the token IDs.
- We fold cyclically to length n with weights = H_2(p) / log(2). Tokens
  drawn at near-deterministic LM steps (p ~= 0 or 1) contribute nearly zero
  to the posterior; tokens drawn at high-entropy steps contribute fully.
- The detection threshold is calibrated by sampling the null distribution
  of the test statistic (random codewords pushed through the same channel)
  and setting threshold = null_mean + z * null_std with z = Phi^-1(1 - fpr).
  This Gaussian-tail calibration is principled because the test statistic
  is a CLT-friendly sum, and it sidesteps the Bernstein bound inside
  Detect, which is over-conservative for entropy-weighted posteriors.
"""

import torch
import numpy as np
from scipy.stats import norm
from prc import KeyGen, Encode, Detect


# -----------------------------------------------------------------------------
# Conversions and entropy
# -----------------------------------------------------------------------------

def signed_to_bits(signed: torch.Tensor) -> torch.Tensor:
    """+/-1 -> {0,1}.  +1 -> 0,  -1 -> 1."""
    return ((1 - signed) / 2).long()


def binary_entropy(p):
    """H_2(p) in nats. Vectorized; safe at p=0 and p=1."""
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0 - 1e-12)
    return -(p * np.log(p) + (1.0 - p) * np.log1p(-p))


# -----------------------------------------------------------------------------
# Generation: now also returns the LM partition probabilities
# -----------------------------------------------------------------------------

def generate_text_watermark_prc(
    model,
    token_ids,
    max_new_tokens,
    encoding_key,
    partition_map,
    eos_token_id=None,
    watermark=True,
):
    """
    Yields (next_token, p1) per step where:
        next_token : (batch, 1) long tensor of generated token IDs
        p1         : (batch,)   float tensor giving LM P[partition 1] at this step
                                (equal to None when watermark=False, since the
                                 detector won't use entropy info on unwatermarked
                                 content -- but we still emit None for symmetry)

    The caller should accumulate both streams: the token stream becomes the
    generated text, and the p1 stream is used at detection time to weight
    each observation by the LM's entropy.
    """
    model.eval()
    # device = token_ids.device

    n = encoding_key[0].shape[0]                    # codeword length

    if watermark:
        print("Watermark Enabled (PRC)", flush=True)
        signed = Encode(encoding_key)               # torch +/-1, length n
        codeword = signed_to_bits(signed).to(device).float()
    else:
        print("Watermark Disabled", flush=True)
        codeword = torch.bernoulli(torch.full((n,), 0.5)).to(device)

    partition_map = partition_map.to(device)        # (2, vocab)

    with torch.no_grad():
        cache = KVCache()
        logits = model(token_ids, cache=cache)[:, -1]
        for pos in range(max_new_tokens):
            logits = model(token_ids)[:, -1]                            # (batch, vocab)
            probs = torch.softmax(logits, dim=-1)
            p1 = (probs * partition_map[1].to(logits.device)).sum(dim=-1)  # (batch,)

            if watermark:
                xi = codeword[pos % n]                                  # 0. or 1.
                bern_p = torch.where(
                    p1 <= 0.5,
                    2 * xi * p1,
                    1 - 2 * (1 - xi) * (1 - p1),
                ).clamp(0.0, 1.0)

                b = torch.bernoulli(bern_p).long()                      # (batch,)
                mask = partition_map[b].to(logits.device)               # (batch, vocab)
                logits = logits.masked_fill(mask == 0, float("-inf"))

            next_token = torch.argmax(logits, dim=-1, keepdim=True)
            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token, p1.detach().cpu()
            token_ids = torch.cat([token_ids, next_token], dim=1)


# -----------------------------------------------------------------------------
# Folding: equal-weight (legacy) and entropy-aware
# -----------------------------------------------------------------------------

def fold_naive(observed_bits: np.ndarray, n: int) -> np.ndarray:
    """Cyclic fold averaging +/-1 signs at each codeword slot."""
    signs = (1 - 2 * observed_bits.astype(np.int64)).astype(np.float64)
    seq_len = signs.shape[0]
    sums = np.zeros(n, dtype=np.float64)
    counts = np.zeros(n, dtype=np.float64)
    idx = np.arange(seq_len) % n
    np.add.at(sums, idx, signs)
    np.add.at(counts, idx, 1)
    return sums / np.maximum(counts, 1.0)


def fold_entropy_weighted(observed_bits: np.ndarray, p_array: np.ndarray,
                          n: int) -> np.ndarray:
    """
    Entropy-weighted cyclic fold. Each observation contributes its sign
    scaled by H_2(p)/log(2), so deterministic LM steps contribute ~0.

    p_array gives the LM's P[partition 1] at each generation step.
    """
    signs = (1 - 2 * observed_bits.astype(np.int64)).astype(np.float64)
    weights = binary_entropy(p_array) / np.log(2)         # in [0, 1]
    seq_len = signs.shape[0]
    sums = np.zeros(n, dtype=np.float64)
    norms = np.zeros(n, dtype=np.float64)
    idx = np.arange(seq_len) % n
    np.add.at(sums, idx, weights * signs)
    np.add.at(norms, idx, weights)
    return sums / np.maximum(norms, 1e-9)


# -----------------------------------------------------------------------------
# Tokens -> bits (the per-step bit observed by the detector)
# -----------------------------------------------------------------------------

def tokens_to_bits(token_ids: torch.Tensor,
                   partition_map: torch.Tensor) -> np.ndarray:
    """Look up each token's partition (0 or 1) -> length-T int array."""
    if token_ids.dim() != 1:
        token_ids = token_ids.flatten()
    bit_for_token = partition_map[1].long().to(token_ids.device)
    bits = bit_for_token[token_ids].detach().cpu().numpy().astype(np.int64)
    return bits


# -----------------------------------------------------------------------------
# Detector test statistic (matches the internals of Detect)
# -----------------------------------------------------------------------------

def _test_statistic(posteriors: np.ndarray, decoding_key) -> float:
    """
    Compute the centered log-likelihood test statistic from Detect's internals,
    so we can compare it to a calibrated threshold.

    Returns log_plus.sum() - 0.5 * log_prod.sum(), which is the centered
    statistic whose null mean is ~0.
    """
    _, parity_check_matrix, one_time_pad, _, noise_rate, _, _, _, t = decoding_key
    pc = (1 - 2 * noise_rate) * (1 - 2 * np.array(one_time_pad, dtype=float)) * posteriors
    r = parity_check_matrix.shape[0]
    Pi = np.prod(pc[parity_check_matrix.indices.reshape(r, t)], axis=1)
    log_plus = np.log(np.clip((1 + Pi) / 2, 1e-15, 1.0))
    log_minus = np.log(np.clip((1 - Pi) / 2, 1e-15, 1.0))
    log_prod = log_plus + log_minus
    return float(log_plus.sum() - 0.5 * log_prod.sum())


# -----------------------------------------------------------------------------
# Threshold calibration
# -----------------------------------------------------------------------------

def calibrate_threshold(
    decoding_key,
    p_array_for_calibration: np.ndarray,
    fpr: float = 1e-9,
    num_calibration_trials: int = 100,
    seed: int = 90210,
) -> dict:
    """
    Estimate the null distribution of the test statistic and return a
    threshold for the requested FPR.

    The null is generated by:
      1. Sampling a random codeword (uniform bits).
      2. Pushing it through the SAME LM channel the user generated text on
         (we reuse p_array_for_calibration to match the entropy profile).
      3. Computing the entropy-weighted folded posterior and its statistic.

    Args:
        decoding_key: from KeyGen.
        p_array_for_calibration: an array of LM partition probabilities to
            reuse when simulating the null. In practice, pass the p1 trace
            recorded during your watermarked generation -- this guarantees
            the null calibration matches the channel statistics of the
            content under test.
        fpr: target false-positive rate.
        num_calibration_trials: how many null samples to draw.

    Returns:
        Dict with keys: 'threshold', 'null_mean', 'null_std', 'z',
        'fpr', 'num_trials'.
    """
    rng = np.random.default_rng(seed)
    n = decoding_key[0].shape[0]
    p_arr = np.asarray(p_array_for_calibration, dtype=np.float64)
    num_tokens = len(p_arr)
    assert num_tokens >= n, (
        f"Need at least n={n} tokens of p-trace for calibration, got "
        f"{num_tokens}. Generate a longer sequence."
    )

    null_stats = np.empty(num_calibration_trials, dtype=np.float64)
    for trial in range(num_calibration_trials):
        # Random "codeword" bits and the resulting realized bits under the
        # same Bernoulli channel.
        random_codeword = rng.integers(0, 2, size=n)
        xi = random_codeword[np.arange(num_tokens) % n]
        bern_p = np.where(p_arr <= 0.5,
                          2 * xi * p_arr,
                          1 - 2 * (1 - xi) * (1 - p_arr))
        bern_p = np.clip(bern_p, 0.0, 1.0)
        observed = rng.binomial(1, bern_p)
        post = fold_entropy_weighted(observed, p_arr, n)
        null_stats[trial] = _test_statistic(post, decoding_key)

    null_mean = float(null_stats.mean())
    null_std = float(null_stats.std(ddof=1))
    z = float(norm.ppf(1.0 - fpr))
    threshold = null_mean + z * null_std

    return {
        "threshold": threshold,
        "null_mean": null_mean,
        "null_std": null_std,
        "z": z,
        "fpr": fpr,
        "num_trials": num_calibration_trials,
    }


# -----------------------------------------------------------------------------
# Detection: entropy-aware + calibrated
# -----------------------------------------------------------------------------

def detect_watermark_prc(
    decoding_key,
    generated_token_ids: torch.Tensor,
    partition_probs: np.ndarray,
    partition_map: torch.Tensor,
    fpr: float = 1e-9,
    num_calibration_trials: int = 100,
    return_info: bool = False,
):
    """
    Entropy-aware, null-calibrated watermark detection.

    Args:
        decoding_key: from KeyGen.
        generated_token_ids: 1-D long tensor of generated token IDs.
        partition_probs: numpy array of the LM's P[partition 1] at each
            generation step. Must have the same length as
            generated_token_ids. Recorded during generate_text_watermark_prc.
        partition_map: (2, vocab) 0/1 indicator tensor.
        fpr: target false-positive rate.
        num_calibration_trials: null calibration sample size.
        return_info: if True, return (decision, info_dict).

    Returns:
        bool decision, or (decision, info_dict) if return_info=True.
    """
    n = decoding_key[0].shape[0]
    bits = tokens_to_bits(generated_token_ids, partition_map)
    p_arr = np.asarray(partition_probs, dtype=np.float64)
    assert bits.shape == p_arr.shape, (
        f"tokens ({bits.shape}) and partition_probs ({p_arr.shape}) "
        f"must have the same length"
    )

    posteriors = fold_entropy_weighted(bits, p_arr, n)
    statistic = _test_statistic(posteriors, decoding_key)

    cal = calibrate_threshold(decoding_key, p_arr, fpr=fpr,
                              num_calibration_trials=num_calibration_trials)

    decision = bool(statistic > cal["threshold"])
    if return_info:
        info = {**cal, "statistic": statistic,
                "sigmas_above_null": (statistic - cal["null_mean"]) / cal["null_std"]
                                    if cal["null_std"] > 0 else float("inf")}
        return decision, info
    return decision



/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


huggingface_hub version: 0.30.2
tokenizers version: 0.21.1
torch version: 2.6.0
Model uses weight tying.
Qwen3-0.6B-Base/tokenizer.json


In [3]:
test_prompts = [
    # Creativity and Writing
    "Write a complete novella-length short story (aim for 8,000+ words) about a generation ship's AI that has been alone for 400 years after the human passengers went into permanent cryosleep. Include the AI's evolving philosophy across centuries, at least five distinct 'eras' of its inner life, encounters with deep-space phenomena, its relationship with the sleeping humans it tends to, invented rituals it develops, and a morally ambiguous ending. Use rich sensory language, interior monologue, and flashbacks to the launch.",
    "Write a comprehensive 5,000+ word craft essay on how to write literary fiction, covering: voice, point of view (first/second/third limited/omniscient with examples of each), scene vs. summary, dialogue mechanics, subtext, image patterning, structure (Freytag, Fichtean, kishōtenketsu, in medias res, frame narratives), revision strategies, and common failure modes. For each topic, include illustrative passages you write yourself, then analyze what each technique accomplishes.",
    "Write the complete first act of a stage play (minimum 40 pages of dialogue and stage directions) about three estranged siblings returning to their childhood home to settle their late mother's estate, only to discover she left behind a series of letters revealing a decades-long secret. Include full character descriptions, set design notes, and at least six scenes with distinct dramatic beats.",
    "Compose 25 original poems on the theme of 'thresholds,' each in a different traditional form: sonnet (Shakespearean), sonnet (Petrarchan), villanelle, sestina, pantoum, ghazal, haiku sequence (10 linked), tanka sequence (10 linked), terza rima, ottava rima, rondeau, triolet, ballade, blank verse monologue, heroic couplets, Spenserian stanza, ode (Pindaric), ode (Horatian), elegy, aubade, eclogue, prose poem, found poem, erasure, and free verse. Before each poem, write a 100-word note on the form's history and constraints.",
    "Draft a complete brand identity and launch document for a fictional company called 'Lumen & Hollow' that sells handmade wooden furniture. Include: founding story (1,000 words), brand manifesto, voice and tone guide with do/don't examples, twelve sample social media posts across platforms, full website copy (homepage, about, three product pages, FAQ, contact), three blog posts (1,500 words each), email welcome sequence (5 emails), packaging copy, and press release for the launch.",
    
    # Logic and Reasoning
    "Provide an exhaustive analysis of the Monty Hall problem. Cover: the original formulation, the intuitive 'wrong' answer and why it's seductive, the correct probabilistic reasoning shown three different ways (enumeration, conditional probability with Bayes, simulation logic), historical reception including the Marilyn vos Savant controversy with specific letters from mathematicians, variations (100 doors, host doesn't know, host is adversarial, two contestants), philosophical implications for epistemology, connections to the Sleeping Beauty problem and the Two Envelopes paradox, and a final pedagogical guide on how to teach it to skeptics.",
    "Walk through, in maximum detail, fifteen famous logical paradoxes (Liar, Russell's, Sorites, Ship of Theseus, Zeno's Achilles and Arrow, Newcomb's, Grandfather, Bootstrap, Berry's, Curry's, Grelling–Nelson, Unexpected Hanging, Gödel's incompleteness as paradox, and Banach–Tarski). For each: history, precise formulation, attempted resolutions across multiple philosophical schools, your assessment of the strongest resolution and why, and implications for logic, math, or metaphysics.",
    "A complex murder mystery: Six suspects, twelve clues, three rooms, and two timelines. Construct the entire mystery from scratch—give each suspect a full biography, motive, alibi, and relationship to the victim and to each other. Then walk a detective through solving it step by step, showing the reasoning chain, every red herring considered and eliminated, every Bayesian update, and the final reconstruction of events. End with a 'how the detective explains it to the suspects in the drawing room' monologue.",
    "Generate 50 highly unconventional but genuinely practical uses for each of the following ordinary objects: a metal paperclip, a rubber band, a brick, a tennis ball, a wire coat hanger, and a dryer sheet. For each use, explain the underlying physical or chemical principle that makes it work, and rate it on a 1–5 scale for practicality, originality, and safety with justification.",
    "Construct a complete decision-theoretic analysis of the question 'Should I go to graduate school?' Build out the full decision tree with at least four levels of branching, assign probability estimates to each node with reasoning, compute expected utilities under three different utility functions (income-maximizing, meaning-maximizing, optionality-maximizing), perform sensitivity analysis on the key parameters, discuss how the answer differs across fields (humanities, STEM, professional), and address objections from behavioral economics about why this kind of analysis often misleads.",
    
    # Coding and Technical
    "Write a complete, production-quality Python implementation of a small toy language interpreter (call it 'Tinyscript'). Include: lexer with full token definitions, recursive descent parser producing an AST, tree-walking interpreter, support for integers, floats, strings, booleans, variables, if/else, while loops, functions with closures, first-class functions, lists, dicts, and a small standard library. Include 30+ unit tests, error handling with helpful messages, a REPL, and 2,000+ words of accompanying documentation explaining each component and the design decisions.",
    "Write an exhaustive guide to concurrency in modern systems, covering: processes vs threads vs fibers vs coroutines vs goroutines, the memory hierarchy and cache coherence, atomics and memory ordering (sequential consistency, acquire-release, relaxed) with C++/Rust code examples, locks (mutex, rwlock, spinlock, ticket lock), lock-free data structures (Treiber stack, Michael-Scott queue) with full implementations, the actor model, CSP, software transactional memory, async/await internals (state machine transformation), event loops, common bugs (data races, deadlocks, livelocks, ABA, false sharing) with diagnostic strategies, and a comparison of Go, Rust, Erlang, and Java approaches. Aim for 8,000+ words with code.",
    "Implement a complete miniature relational database in Python from scratch (no external libraries beyond the standard library). Include: a B+ tree index with insert/delete/range query, a heap file storage engine, a buffer pool with LRU eviction, a SQL parser supporting SELECT/INSERT/UPDATE/DELETE/CREATE TABLE/JOIN, a query planner that chooses between index and sequential scan, a basic transaction manager with two-phase locking, write-ahead logging with crash recovery, and a CLI. Include thorough comments and a 3,000+ word architectural overview.",
    "Write a complete tutorial on training a small transformer language model from scratch, suitable for someone who has used PyTorch but never trained an LM. Cover: tokenization (BPE from scratch with code), the data pipeline (streaming, packing, masking), the architecture (multi-head attention with derivation, RMSNorm, SwiGLU, RoPE with mathematical motivation), training loop (mixed precision, gradient accumulation, learning rate schedules, gradient clipping), distributed training options (DDP, FSDP, tensor parallelism, pipeline parallelism with diagrams), evaluation (perplexity, downstream tasks), debugging methodology (loss spikes, NaN diagnosis, gradient flow inspection), and inference optimizations (KV cache, speculative decoding). Include working code throughout.",
    "Pick fifteen classic algorithms and data structures (Dijkstra, A*, Floyd-Warshall, Union-Find with path compression, Fenwick tree, segment tree with lazy propagation, suffix array with LCP, KMP, Aho-Corasick, treap, skip list, persistent segment tree, link-cut tree, max-flow with Dinic's, bipartite matching with Hopcroft-Karp). For each: motivating problem, full implementation in C++ or Python with comments, complexity analysis with proof sketches, common pitfalls, and three example problems where it's the right tool with full solutions.",
    
    # Factual and Educational
    "Write a comprehensive 10,000-word history of the French Revolution from the financial crisis of the 1770s through the rise of Napoleon. Cover: structural causes (fiscal, agrarian, intellectual), the Estates-General, the storming of the Bastille, the Great Fear, the women's march on Versailles, the Civil Constitution of the Clergy, the flight to Varennes, the war with Austria, the September Massacres, the Convention, the trial and execution of Louis XVI, the Terror with biographical sketches of Robespierre/Saint-Just/Danton, Thermidor, the Directory, and the 18 Brumaire. Embed primary source quotations, address competing historiographical traditions (Marxist, revisionist, post-revisionist), and end with the Revolution's contested legacy.",
    "Provide a complete tour of cell biology suitable for a motivated undergraduate. Cover: cell theory and its history, prokaryote vs. eukaryote architecture, every major organelle in detail (nucleus, ER rough and smooth, Golgi, mitochondria with the endosymbiotic hypothesis evidence, chloroplasts, lysosomes, peroxisomes, cytoskeleton with all three filament types), the plasma membrane and transport (passive, facilitated, active, vesicular), the cell cycle with checkpoint biology, mitosis and meiosis with detailed phase comparison, signaling pathways (GPCR, RTK, JAK-STAT, Wnt, Notch, Hedgehog), apoptosis vs. necrosis vs. ferroptosis, and how cancer dysregulates each system. 8,000+ words.",
    "Write an in-depth exploration of the Fermi Paradox. Begin with Fermi's original lunchtime question and Hart's 1975 formalization. Then walk through twenty proposed solutions in detail: the Rare Earth hypothesis, the Great Filter (early vs. late), the Zoo hypothesis, the Dark Forest, the Berserker hypothesis, the simulation hypothesis as resolution, the transcension hypothesis, the firstborn hypothesis, communication mismatch (timescale, frequency, modality), self-destruction filters, the planetarium hypothesis, postbiological evolution, the aestivation hypothesis, the percolation theory of colonization, sublight expansion economics, and several more. For each: the steelmanned argument, empirical constraints, and how it could in principle be tested.",
    "Write a complete, accessible explanation of general relativity. Begin with the equivalence principle and Einstein's elevator thought experiment, build up tensor calculus from scratch (vectors, dual vectors, tensors, the metric, covariant derivative, Christoffel symbols, Riemann curvature, Ricci tensor, Ricci scalar), derive the Einstein field equations, then solve for: the Schwarzschild metric (with derivation), perihelion precession of Mercury, gravitational lensing, gravitational time dilation, the Kerr metric (qualitatively), gravitational waves (linearized theory), and FLRW cosmology. Include diagrams described in detail and worked numerical examples. 10,000+ words.",
    "Write an exhaustive history of the Australian capital question, from federation in 1901 through the design competition for Canberra, Walter Burley Griffin's plan, the construction phases, the move of government in 1927, and Canberra's evolution as a planned city. Embed this in the broader history of capital city selection worldwide, comparing with Brasília, Washington D.C., Ottawa, Abuja, Astana, and Naypyidaw. Conclude with a discussion of what makes a planned capital succeed or fail.",
    
    # Formatting and Constraints
    "Generate a richly detailed JSON document representing a fictional library's complete catalog: 30 books across 6 genres, each with title, author (with full nested author objects including birth/death years, nationality, bibliography), publication year, ISBN, publisher, page count, language, original language, translator (where applicable), genres (array), tags (array), summary (100+ words), 3 fictional reader reviews each (with reviewer name, rating, date, full review text), checkout history (10 entries each), and a 'related works' array. Then describe the schema in a separate section.",
    "Produce a single comprehensive Markdown document comparing 30 programming languages across 25 dimensions (typing discipline, memory management, paradigm support, concurrency model, performance tier, ecosystem maturity, learning curve, primary domains, notable companies using it, killer feature, biggest weakness, year created, creator, governance model, package manager, build system, REPL availability, metaprogramming support, error handling style, null handling, generics support, FFI quality, mobile support, web support, embedded support). Use a large table, then write 200-word profiles of each language afterward.",
    "Summarize the entire plot of Shakespeare's Hamlet at twelve different lengths, in this exact order: 1 sentence, 3 sentences, 1 paragraph, 3 paragraphs, 500 words, 1,000 words, 2,000 words act-by-act, 4,000 words scene-by-scene, a 6,000-word version that includes character analysis, a 10,000-word version that includes thematic analysis and historical context, a one-page bullet outline, and a single 17-syllable haiku. Then write a 2,000-word reflection on what is preserved and what is lost at each length.",
    "Produce an alphabetized encyclopedia of every named celestial body in our solar system: all eight planets, all five officially recognized dwarf planets, all major moons (at least the top 25 by size), the largest asteroids, the most significant trans-Neptunian objects, and the most studied comets. For each entry: alphabetized header, discovery date and discoverer, etymology of name, physical properties (mass, radius, density, surface gravity, escape velocity, rotation period, orbital period, semi-major axis, eccentricity, inclination), composition, atmosphere, notable surface features, exploration history with mission list, and 200+ words of narrative description.",
    "Translate the phrase 'Hello, how are you doing today? I hope you are well' into 30 languages: French, Spanish, Italian, Portuguese (Brazilian and European, separately), German, Dutch, Swedish, Norwegian, Danish, Finnish, Polish, Russian, Ukrainian, Czech, Mandarin (simplified and traditional), Cantonese, Japanese (with kanji, hiragana, and romaji), Korean (with hangul and romanization), Vietnamese, Thai, Hindi, Bengali, Arabic (Modern Standard), Hebrew, Turkish, Swahili, Zulu, and Esperanto. For each, also provide: phonetic pronunciation guide, morphological breakdown, register notes (formal vs. casual variations), cultural notes on greeting customs, and one common response.",
    
    # Roleplay and Persona
    "Write a 6,000+ word stream-of-consciousness monologue from Leonardo da Vinci, transported to the present day, walking through a modern airport, a hospital, a smartphone repair shop, a particle physics lab, an art museum showing contemporary art, and a fast food restaurant. He should react in his own voice (thoughtful, omnivorous curiosity, polymath cross-references, occasional Tuscan exclamation), connect what he sees to his own notebooks and inventions, critique and praise, propose modifications, and have one moment of genuine awe and one of genuine sorrow. End with a letter he writes back to Ludovico Sforza.",
    "You are a senior travel concierge with 25 years of experience. Build a complete, day-by-day, hour-by-hour 14-day itinerary for a couple in their late thirties traveling through Japan in late October, with a moderate budget, interests in food/architecture/contemporary art/hiking, and a willingness to take overnight ryokan stays. Cover Tokyo, Hakone, Kyoto, Naoshima, Kanazawa, and a final return. For each day: morning/afternoon/evening blocks with specific venues, reservations needed, transit instructions with exact train lines and approximate costs, restaurant recommendations at three price tiers per meal, alternate plans for rain, etiquette notes, useful phrases, and packing reminders. Include a separate 2,000-word pre-trip preparation guide.",
    "Write a 10,000-word philosophical dialogue in the style of Plato, between five characters representing different ethical traditions (a virtue ethicist channeling Aristotle, a Kantian deontologist, a utilitarian, a contractualist, and a moral particularist), debating whether it is permissible to lie to a murderer at the door asking where their intended victim is hiding. Each character should make their strongest case, respond to the others, refine their position under pressure, and at least one should change their mind in a philosophically interesting way. Include stage directions and a final synthesis offered by a sixth character (a moral skeptic) that everyone finds unsatisfying for instructive reasons.",
    "Adopt the persona of a meticulous 1890s natural historian writing a field guide. Document twenty fictional species of magical creatures inhabiting a temperate forest. For each species: scientific binomial in faux-Latin with etymology, common names in three languages, full physical description (300+ words), habitat and range with maps described in prose, diet, mating and reproduction, social behavior, observed intelligence, magical properties with proposed mechanisms, historical encounters and folklore, conservation status, and the author's personal field notes from an encounter, complete with weather, time of day, and the author's emotional reaction. Use period-appropriate prose throughout.",
    "Assume the persona of an 18th-century ship's surgeon keeping a daily log during an 18-month voyage. Write 60 dated journal entries (averaging 400 words each) covering: departure, the first storm, an outbreak of scurvy and your treatment attempts, conflict with the captain, an encounter with another vessel, a man overboard, the crossing of the equator with its rituals, a stop at a Pacific island and your ethnographic and botanical observations, a mutiny attempt, a stretch of becalmed days where you record philosophical reflections, the eventual return, and a final entry written years later. Maintain consistent voice, evolving relationships with named crew members, and accurate period medicine.",
]

In [4]:
# --- One-time setup --------------------------------------------------------
encoding_key, decoding_key = KeyGen(
    n=4096, message_length=0,
    false_positive_rate=0.5,    # unused now; calibration sets the threshold
    noise_rate=0.05,            # eta the encoder injects
)



In [8]:
# -----------------------------------------------------------------------------
# Phase 1: helper to record token IDs and p-traces during generation
# -----------------------------------------------------------------------------

def generate_and_collect(generator):
    """
    Drains a `generate_text_watermark_prc` generator into two lists:
    a flat 1-D long tensor of token IDs and a 1-D float array of p1 values.

    Pass the iterator object directly:
        gen = generate_text_watermark_prc(model, prompt_ids, ..., partition_map=pm)
        tokens, p_trace = generate_and_collect(gen)
    """
    tok_chunks, p_chunks = [], []
    for next_token, p1 in generator:
        tok_chunks.append(next_token)                  # (batch, 1)
        p_chunks.append(p1)                            # (batch,)

    if not tok_chunks:
        return torch.zeros(0, dtype=torch.long), np.zeros(0, dtype=np.float64)

    tokens = torch.cat(tok_chunks, dim=1).flatten()    # assumes batch=1
    p_trace = torch.stack(p_chunks).flatten().numpy().astype(np.float64)
    return tokens, p_trace


# -----------------------------------------------------------------------------
# Phase 2: calibration
# -----------------------------------------------------------------------------

def fit_calibration(
    decoding_key,
    calibration_p_traces,
    fpr: float = 1e-9,
    num_simulated_nulls: int = 2000,
    min_trace_length: int | None = None,
    seed: int = 1234,
) -> dict:
    """
    Fit a single detection threshold from a batch of p-traces.

    For each of `num_simulated_nulls` synthetic null trials we:
      1. Pick a random p-trace from the calibration set.
      2. Truncate or accept it (must be >= n long, since the detector cycles
         the codeword over each n-window).
      3. Sample a fresh random codeword and run the Bernoulli channel.
      4. Compute the entropy-weighted test statistic.

    The threshold is then null_mean + Phi^-1(1 - fpr) * null_std.

    Args:
        decoding_key: from KeyGen.
        calibration_p_traces: a list (or iterable) of 1-D numpy arrays of
            partition probabilities, one per calibration generation.
        fpr: target false-positive rate.
        num_simulated_nulls: total number of synthetic null statistics.
        min_trace_length: drop any calibration trace shorter than this.
            Defaults to n (the codeword length).
        seed: RNG seed.

    Returns:
        dict with 'threshold', 'null_mean', 'null_std', 'fpr', 'z',
        'n', 'num_traces_used', 'num_simulated_nulls'.
    """
    n = decoding_key[0].shape[0]
    if min_trace_length is None:
        min_trace_length = n

    traces = [np.asarray(p, dtype=np.float64)
              for p in calibration_p_traces
              if len(p) >= min_trace_length]
    if not traces:
        raise ValueError(
            f"No calibration traces of length >= {min_trace_length}. "
            f"Generate longer sequences or lower min_trace_length."
        )

    rng = np.random.default_rng(seed)
    null_stats = np.empty(num_simulated_nulls, dtype=np.float64)

    for i in range(num_simulated_nulls):
        p_arr = traces[rng.integers(0, len(traces))]
        T = len(p_arr)

        codeword = rng.integers(0, 2, size=n)
        xi = codeword[np.arange(T) % n]
        bern_p = np.where(p_arr <= 0.5,
                          2 * xi * p_arr,
                          1 - 2 * (1 - xi) * (1 - p_arr))
        bern_p = np.clip(bern_p, 0.0, 1.0)
        observed = rng.binomial(1, bern_p)

        post = fold_entropy_weighted(observed, p_arr, n)
        null_stats[i] = _test_statistic(post, decoding_key)

    null_mean = float(null_stats.mean())
    null_std = float(null_stats.std(ddof=1))
    z = float(norm.ppf(1.0 - fpr))
    threshold = null_mean + z * null_std

    return {
        "threshold": threshold,
        "null_mean": null_mean,
        "null_std": null_std,
        "fpr": fpr,
        "z": z,
        "n": n,
        "num_traces_used": len(traces),
        "num_simulated_nulls": num_simulated_nulls,
    }


def save_threshold_state(state: dict, path: str) -> None:
    """Persist a calibration result to JSON."""
    with open(path, "w") as f:
        json.dump(state, f, indent=2)


def load_threshold_state(path: str) -> dict:
    """Load a calibration result from JSON."""
    with open(path) as f:
        return json.load(f)


# -----------------------------------------------------------------------------
# Phase 3: detection without recalibration
# -----------------------------------------------------------------------------

def detect_with_threshold(
    decoding_key,
    generated_token_ids: torch.Tensor,
    partition_probs: np.ndarray,
    partition_map: torch.Tensor,
    threshold_state: dict,
    return_info: bool = False,
):
    """
    Fast detection using a precomputed threshold.

    Args:
        decoding_key: from KeyGen.
        generated_token_ids: 1-D long tensor of generated token IDs.
        partition_probs: 1-D float array, same length as generated_token_ids,
            recorded during generation.
        partition_map: (2, vocab) 0/1 indicator tensor.
        threshold_state: dict returned by fit_calibration (or loaded from disk).
        return_info: if True, also return a dict with statistic and sigmas.

    Returns:
        bool decision, or (decision, info) if return_info=True.
    """
    n = decoding_key[0].shape[0]
    if threshold_state["n"] != n:
        raise ValueError(
            f"Calibration was for n={threshold_state['n']}, "
            f"but decoding_key has n={n}."
        )

    bits = tokens_to_bits(generated_token_ids, partition_map)
    p_arr = np.asarray(partition_probs, dtype=np.float64)
    if bits.shape != p_arr.shape:
        raise ValueError(
            f"tokens length {bits.shape[0]} != partition_probs length {p_arr.shape[0]}"
        )

    posteriors = fold_entropy_weighted(bits, p_arr, n)
    statistic = _test_statistic(posteriors, decoding_key)

    decision = bool(statistic > threshold_state["threshold"])
    if return_info:
        sigmas = ((statistic - threshold_state["null_mean"])
                  / threshold_state["null_std"]) if threshold_state["null_std"] > 0 \
                  else float("inf")
        return decision, {
            "statistic": statistic,
            "threshold": threshold_state["threshold"],
            "sigmas_above_null": sigmas,
        }
    return decision

In [12]:
prompt_ids

tensor([151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
        151645,    198, 151644,    872,    198,   7985,    264,   4583,   6609,
          6842,  29325,   2805,   3364,    320,   2640,    369,    220,     23,
            11,     15,     15,     15,     10,   4244,      8,    911,    264,
          9471,   8284,    594,  15235,    429,    702,   1012,   7484,    369,
           220,     19,     15,     15,   1635,   1283,    279,   3738,  22172,
          3937,   1119,  15330,  15729,    436,   3499,     13,  29734,    279,
         15235,    594,  40928,  19128,   3941,  23631,     11,    518,   3245,
          4236,  12460,    364,   9247,      6,    315,   1181,   9179,   2272,
            11,  33906,    448,   5538,  27947,  43147,     11,   1181,   5025,
           448,    279,  21127,  12677,    432,  27236,    311,     11,  35492,
         56579,    432,  38571,     11,    323,    264,  56223,  54761,  13391,
            13,   5443,   9080,  47969, 

In [13]:
# --- Phase 1: collect calibration p-traces from your dataset ---------------
calibration_traces = []
for prompt in test_prompts:
    prompt_ids = torch.Tensor([prompt_to_ids(prompt)]).long().to(device)
    gen = generate_text_watermark_prc(
        model, prompt_ids, max_new_tokens=4*4096,
        encoding_key=encoding_key, partition_map=partition,
        eos_token_id=tokenizer.eos_token_id,
    )
    _, p_trace = generate_and_collect(gen)
    calibration_traces.append(p_trace)



Watermark Enabled (PRC)


KeyboardInterrupt: 

In [14]:
calibration_traces

[]

In [ ]:
# --- Phase 2: fit threshold once and persist -------------------------------
threshold_state = fit_calibration(
    decoding_key, calibration_traces,
    fpr=1e-9, num_simulated_nulls=2000,
)
save_threshold_state(threshold_state, "qwen_threshold.json")


In [ ]:

# --- Phase 3: production. Generate + detect with the precomputed threshold -
threshold_state = load_threshold_state("qwen_threshold.json")
for prompt_ids in production_prompts:
    gen = generate_text_watermark_prc(
        model, prompt_ids, max_new_tokens=4*4096,
        encoding_key=encoding_key, partition_map=partition_map,
    )
    tokens, p_trace = generate_and_collect(gen)

    detected, info = detect_with_threshold(
        decoding_key, tokens, p_trace, partition_map,
        threshold_state, return_info=True,
    )
    print(f"detected={detected}, separation={info['sigmas_above_null']:.1f}σ")

In [2]:

from prc import KeyGen

encoding_key, decoding_key = KeyGen(
    n=4096, message_length=0,
    false_positive_rate=0.5,   # only used by Detect's internal threshold,
                               # which we don't rely on anymore
    noise_rate=0.05,           # eta the encoder injects
)


for prompt in test_prompts[:5]:
    prompt_ids = prompt_to_ids(prompt)
    prompt_ids = torch.Tensor([prompt_ids]).long().to(device)
    print(prompt_ids)
    tokens, p_trace = [], []
    for token, p1 in generate_text_watermark_prc(
        model, prompt_ids, max_new_tokens=4096,   # 4n tokens recommended
        encoding_key=encoding_key, partition_map=partition,
        eos_token_id=tokenizer.eos_token_id):
        tokens.append(token)
        p_trace.append(p1)
    generated = torch.cat(tokens, dim=1).flatten()
    p_array = torch.cat(p_trace).float().numpy()
    decision, info = detect_watermark_prc(
            decoding_key, generated, p_array, partition,
            fpr=1e-9, return_info=True,
        )
    print(f"Detected: {decision}  ({info['sigmas_above_null']:.1f} sigma above null)")
    break

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   7985,    264,   6386,  38242,
            911,    264,  12305,   6832,    311,   2666,  21261,     13, 151645,
            198, 151644,  77091,    198]], device='cuda:0')
Watermark Enabled (PRC)


AssertionError: Need at least n=256 tokens of p-trace for calibration, got 11. Generate a longer sequence.